# 3.1 Steady-State Simulation of Models and Media

## Import the libraries
- COBRApy

In [1]:
import cobra as cb

## Load the models

GEMs Files available at [vmh.life](https://www.vmh.life/files/reconstructions/AGORA2/version2.01/mat_files/individual_reconstructions/) or directly:

[AGORA 2 Files - Eubacterium hallii DSM 3353](https://www.vmh.life/files/reconstructions/AGORA2/version2.01/mat_files/individual_reconstructions/Eubacterium_hallii_DSM_3353.mat) 

[AGORA 2 Files - Bifidobacterium longum infantis ATCC 15697](https://www.vmh.life/files/reconstructions/AGORA2/version2.01/mat_files/individual_reconstructions/Bifidobacterium_longum_infantis_ATCC_15697.mat) 

In [2]:
model_path = 'Bifidobacterium_longum_infantis_ATCC_15697.mat'
B_infantis_model = cb.io.load_matlab_model( model_path)
model_path = 'Eubacterium_hallii_DSM_3353.mat'
A_hallii_model = cb.io.load_matlab_model(model_path)

Set parameter Username
Academic license - for non-commercial use only - expires 2027-02-19


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e


## Define the base medium

In [3]:
base_medium = {
    'EX_o2(e)': 0,           # anaerobic
    'EX_h2o(e)': 1000,       # water
    'EX_pi(e)': 1000,        # phosphate
    'EX_fe2(e)': 1000,       # ferrous iron
    'EX_fe3(e)': 1000,       # ferric iron
    'EX_zn2(e)': 1000,       # zinc
    'EX_so4(e)': 1000,       # sulfate
    'EX_cu2(e)': 1000,       # copper
    'EX_k(e)': 1000,         # potassium
    'EX_mg2(e)': 1000,       # magnesium
    'EX_mn2(e)': 1000,       # manganese
    'EX_cd2(e)': 1000,       # cadmium
    'EX_cl(e)': 1000,        # chloride
    'EX_ca2(e)': 1000,       # calcium
    'EX_cobalt2(e)': 1000,   # cobalt
    'EX_glc_D(e)': 10,       # glucose (10 mM)
    'EX_nh4(e)': 20}          # ammonium

## Find the minimal medium for B. infantis

In [4]:
# Find minimal media with specific components always added
with B_infantis_model as model_t:
    for rxn_id, uptake in base_medium.items():
        met = list(model_t.exchanges.get_by_id(rxn_id).metabolites.keys())[0]
        model_t.add_boundary(met, type="sink", reaction_id=rxn_id+'_tmp',lb=-1*uptake,ub=1000)
        
    for ex in model_t.exchanges:
        ex.lower_bound = -1000
        ex.upper_bound = 1000

    mm = cb.medium.minimal_medium(model_t,0.1,minimize_components=10)
print(mm)

                     0         1         2         3         4
EX_alahis(e)  0.892649  0.009262  0.009262  0.000000  0.000000
EX_asn_L(e)   0.000000  0.023468  0.023468  0.000000  0.023468
EX_cgly(e)    0.000000  0.000000  0.026175  0.026175  0.026175
EX_glyasn(e)  0.104065  0.000000  0.000000  2.994995  0.000000
EX_glycys(e)  0.026175  0.026175  0.000000  0.000000  0.000000
EX_his_L(e)   0.000000  0.000000  0.000000  0.009262  0.009262
EX_nac(e)     0.001562  0.001562  0.001562  0.001562  0.000000
EX_o2(e)      0.000000  0.000000  0.000000  0.000000  0.001562
EX_pnto_R(e)  0.001562  0.001562  0.001562  0.001562  0.001562
EX_ribflv(e)  0.001562  0.001562  0.001562  0.001562  0.001562


In [5]:
# define the new medium
new_medium_components = {
    'EX_ribflv(e)': 1000,
    'EX_pnto_R(e)': 1000,
    'EX_nac(e)':1000,
    'EX_his_L(e)':1000,
    'EX_asn_L(e)':1000,
    'EX_glycys(e)': 1000,
}
new_medium = base_medium | new_medium_components


## Confirm B. infantis growth

In [6]:
with B_infantis_model as model_t:
    for exchange in model_t.exchanges:
        exchange.upper_bound = 1000
        if exchange.id in new_medium:
            exchange.lower_bound = -1 * new_medium[exchange.id]
        else:
            exchange.lower_bound = 0
    sol = model_t.optimize()
    print(sol.objective_value)
    print(sol.fluxes['EX_lac_L(e)'])

0.10966679074428674
18.383776726251586


## Identify additional medium components for A. hallii growth

In [7]:
L_lactate_medium = {'EX_lac_L(e)': 10}
base_medium_2 = new_medium | L_lactate_medium

# Find minimal media with specific components always added
with A_hallii_model as model_t:
    for rxn_id, uptake in base_medium_2.items():
        met = list(model_t.exchanges.get_by_id(rxn_id).metabolites.keys())[0]
        model_t.add_boundary(met, type="sink", reaction_id=rxn_id+'_tmp',lb=-1*uptake,ub=1000)
        
    for ex in model_t.exchanges:
        ex.lower_bound = -1000
        ex.upper_bound = 1000

    mm = cb.medium.minimal_medium(model_t,0.1,minimize_components=10)
    print(mm)

                     0         1         2         3
EX_alagly(e)  0.051144  0.066883  0.059891  0.146717
EX_alaleu(e)  0.000000  0.000000  0.043866  0.043866
EX_glyleu(e)  0.043866  0.000000  0.000000  0.000000
EX_glymet(e)  0.000000  0.015715  0.015715  0.000000
EX_hxan(e)    0.045736  0.045736  0.045736  0.045736
EX_leu_L(e)   0.000000  0.043866  0.000000  0.000000
EX_lys_L(e)   0.033355  0.033355  0.033355  0.033355
EX_met_L(e)   0.000000  0.000000  0.000000  0.015715
EX_metala(e)  0.015715  0.000000  0.000000  0.000000


## Define final medium

In [8]:
# define the final medium
final_medium_components = {
    'EX_lys_L(e)': 1000,
    'EX_ala_L(e)': 1000,
    'EX_met_L(e)': 1000,
    'EX_leu_L(e)': 1000,
    'EX_hxan(e)': 1000,    
    'EX_glyasn(e)': 1000
}
final_medium = base_medium_2 | final_medium_components

# Confirm growth of both organisms on final medium

In [9]:
# check B. infantis growth on the final medium
with B_infantis_model as model_t:
    for exchange in model_t.exchanges:
        exchange.upper_bound = 1000
        if exchange.id in final_medium:
            exchange.lower_bound = -1 * final_medium[exchange.id]
        else:
            exchange.lower_bound = 0
    sol = model_t.optimize()
    print('B_infantis objective value:', sol.objective_value)

# check A. hallii growth on the final medium
with A_hallii_model as model_t:
    for exchange in model_t.exchanges:
        exchange.upper_bound = 1000
        if exchange.id in final_medium:
            exchange.lower_bound = -1 * final_medium[exchange.id]
        else:
            exchange.lower_bound = 0
    sol = model_t.optimize()
    print('A_hallii objective value:', sol.objective_value)

B_infantis objective value: 0.1257347330606084
A_hallii objective value: 0.4841839389036064


In [10]:
import pandas as pd

# Create a DataFrame from final_medium
df = pd.DataFrame(list(final_medium.items()), columns=['Exchange_Reaction', 'Uptake_Bound'])

# Extract metabolite ID from exchange ID (EX_xxx(e) -> xxx[e])
df['Metabolite_ID'] = df['Exchange_Reaction'].str.replace(r'EX_(.+)\((.)\)', r'\1[\2]', regex=True)

# Look up official metabolite names from the model
metabolite_names = []
for met_id in df['Metabolite_ID']:
    try:
        metabolite = B_infantis_model.metabolites.get_by_id(met_id)
        metabolite_names.append(metabolite.name)
    except KeyError:
        try:
            metabolite = A_hallii_model.metabolites.get_by_id(met_id)
            metabolite_names.append(metabolite.name)
        except KeyError:
            metabolite_names.append('Unknown')

df['Metabolite_Name'] = metabolite_names

# Add notes column explaining why each metabolite was added
notes_dict = {
    'EX_o2(e)': 'Anaerobic',
    'EX_h2o(e)': 'Base',
    'EX_pi(e)': 'Base',
    'EX_fe2(e)': 'Base',
    'EX_fe3(e)': 'Base',
    'EX_zn2(e)': 'Base',
    'EX_so4(e)': 'Base',
    'EX_cu2(e)': 'Base',
    'EX_k(e)': 'Base',
    'EX_mg2(e)': 'Base',
    'EX_mn2(e)': 'Base',
    'EX_cd2(e)': 'Base',
    'EX_cl(e)': 'Base',
    'EX_ca2(e)': 'Base',
    'EX_cobalt2(e)': 'Base',
    'EX_glc_D(e)': 'Base',
    'EX_nh4(e)': 'Base',
    'EX_ribflv(e)': 'B. infantis',
    'EX_pnto_R(e)': 'B. infantis',
    'EX_nac(e)': 'B. infantis',
    'EX_his_L(e)': 'B. infantis',
    'EX_asn_L(e)': 'B. infantis',
    'EX_glycys(e)': 'B. infantis',
    'EX_lac_L(e)': 'Secreted by B. infantis',
    'EX_lys_L(e)': 'A. hallii',
    'EX_ala_L(e)': 'A. hallii',
    'EX_met_L(e)': 'A. hallii',
    'EX_leu_L(e)': 'A. hallii',
    'EX_hxan(e)': 'A. hallii',
    'EX_glyasn(e)': 'A. hallii'
}

df['Notes'] = df['Exchange_Reaction'].map(notes_dict).fillna('Unknown')

# Reorder columns for better readability
df = df[['Exchange_Reaction', 'Metabolite_ID', 'Metabolite_Name', 'Uptake_Bound', 'Notes']]

# Save to CSV
df.to_csv('final_medium.csv', index=False)

print(df)

   Exchange_Reaction Metabolite_ID      Metabolite_Name  Uptake_Bound  \
0           EX_o2(e)         o2[e]                   O2             0   
1          EX_h2o(e)        h2o[e]                Water          1000   
2           EX_pi(e)         pi[e]    hydrogenphosphate          1000   
3          EX_fe2(e)        fe2[e]                 Fe2+          1000   
4          EX_fe3(e)        fe3[e]                 Fe3+          1000   
5          EX_zn2(e)        zn2[e]                 Zinc          1000   
6          EX_so4(e)        so4[e]              sulfate          1000   
7          EX_cu2(e)        cu2[e]                 Cu2+          1000   
8            EX_k(e)          k[e]            potassium          1000   
9          EX_mg2(e)        mg2[e]            magnesium          1000   
10         EX_mn2(e)        mn2[e]                 Mn2+          1000   
11         EX_cd2(e)        cd2[e]              Cadmium          1000   
12          EX_cl(e)         cl[e]             Chlo